<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/05_cryosparc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## ⭐ Setup
You must run all codes under this category.

In [1]:
# @title packages imports

import os
import subprocess
import json
import socket
import re
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

### ✅ Directory Settings

In [2]:
# @title  { display-mode: "form" }
# @markdown Directory parameters.

DATA_DIR = "/content/dataset" # @param {type:"string"}
Input_MICROGRAPH_DATA_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output" # @param {type: "string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_no_DT/10017" # @param {type:"string"}
STAR_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_no_DT/10017/unet_eb5_dice_CRF" # @param {type:"string"}
TMP_DIR = "/content/tmp" # @param {type:"string"}
SOFTWARE_DIR = "/content/software" # @param {type:"string"}
PROJECT_DIR = "/content/sparc_project" # @param {type:"string"}
WORK_DIR = os.getcwd()

In [3]:
# @title  { display-mode: "form" }
# @markdown Run this block if using folder📁 in Google Drive as **`RESULT_DIR`**.

try:
  from google.colab import drive
  drive.mount('/content/drive')
except:
  pass

Mounted at /content/drive


In [4]:
# @title  { display-mode: "form" }
# @markdown Run this block for remove the **`sample_data`** folder📁 in content

if os.path.isdir("/content/sample_data"):
  !rm -r /content/sample_data
# from shutil import rmtree
#
# rmtree(f"/content/sample_data")

In [5]:
# @title  { display-mode: "form" }
# @markdown Run this block for checking the existence of the directories

if not os.path.isdir(DATA_DIR):
  os.mkdir(DATA_DIR)

if not os.path.isdir(RESULT_DIR):
  os.mkdir(RESULT_DIR)

if not os.path.isdir(TMP_DIR):
  os.mkdir(TMP_DIR)

if not os.path.isdir(SOFTWARE_DIR):
  os.mkdir(SOFTWARE_DIR)


if not os.path.isdir(PROJECT_DIR):
  os.mkdir(PROJECT_DIR)

### ✅ EMPIAR Data Setting

In [6]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}

In [7]:
EMPIAR_DIR = os.path.join(DATA_DIR, f"EMPIAR-{EMPIAR_ID}")
!mkdir {EMPIAR_DIR}

print((EMPIAR_ID, DATA_DIR, RESULT_DIR, EMPIAR_DIR))

(10017, '/content/dataset', '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_no_DT/10017', '/content/dataset/EMPIAR-10017')


In [8]:
# @title Copy Data to `->/content/data`
!mkdir {EMPIAR_DIR}/micrographs

#!rsync -av {Input_MICROGRAPH_DATA_DIR}/dataset/{EMPIAR_ID}/ground_truth/  {EMPIAR_DIR}/ground_truth/
!rsync -av {Input_MICROGRAPH_DATA_DIR}/dataset/{EMPIAR_ID}/micrographs/ {EMPIAR_DIR}/micrographs/
#!rsync -av {Input_MICROGRAPH_DATA_DIR}/dataset/{EMPIAR_ID}/particles_stack/ {EMPIAR_DIR}/particles_stack/

# !cp {STAR_DIR}/cv_particles.star {EMPIAR_DIR}/
# !cp {STAR_DIR}/tp_particles.star {EMPIAR_DIR}/
# !cp {STAR_DIR}/nms_particles.star {EMPIAR_DIR}/

sending incremental file list
./
Falcon_2012_06_12-14_33_35_0.mrc
Falcon_2012_06_12-14_57_34_0.mrc
Falcon_2012_06_12-15_07_41_0.mrc
Falcon_2012_06_12-15_14_01_0.mrc
Falcon_2012_06_12-15_17_31_0.mrc
Falcon_2012_06_12-15_27_22_0.mrc
Falcon_2012_06_12-15_30_21_0.mrc
Falcon_2012_06_12-15_33_42_0.mrc
Falcon_2012_06_12-15_36_26_0.mrc
Falcon_2012_06_12-15_41_22_0.mrc
Falcon_2012_06_12-15_43_48_0.mrc
Falcon_2012_06_12-15_46_37_0.mrc
Falcon_2012_06_12-15_53_09_0.mrc
Falcon_2012_06_12-16_26_22_0.mrc
Falcon_2012_06_12-16_44_07_0.mrc
Falcon_2012_06_12-16_48_06_0.mrc
Falcon_2012_06_12-16_51_03_0.mrc
Falcon_2012_06_12-16_55_40_0.mrc
Falcon_2012_06_12-16_59_12_0.mrc
Falcon_2012_06_12-17_02_43_0.mrc
Falcon_2012_06_12-17_14_17_0.mrc
Falcon_2012_06_12-17_17_05_0.mrc
Falcon_2012_06_12-17_23_32_0.mrc
Falcon_2012_06_12-17_26_54_0.mrc
Falcon_2012_06_12-17_34_38_0.mrc
Falcon_2012_06_12-17_37_29_0.mrc
Falcon_2012_06_12-17_40_47_0.mrc
Falcon_2012_06_12-17_46_08_0.mrc
Falcon_2012_06_12-17_49_17_0.mrc
Falcon_201

---

#### 🟪 CryoSPARC Installation

In [9]:
# @title  { display-mode: "form" }

INSTALL = True # @param {type:"boolean"}

In [10]:
# @title  { display-mode: "form" }
# @markdown Download and extract CryoSPARC.

if INSTALL:
  try:
    LICENSE_ID = '9657b740-93cb-11f0-a34d-cb5f4bdb076d' # @param {type:"string"}
  except:
    pass

  os.environ['LICENSE_ID'] = LICENSE_ID

  %cd {SOFTWARE_DIR}
  %mkdir cryosparc
  %cd cryosparc
  if not os.path.isdir("cryosparc_master"):
    if not os.path.exists("cryosparc_master.tar.gz"):
      !curl -L https://get.cryosparc.com/download/master-v4.7.1/$LICENSE_ID -o cryosparc_master.tar.gz
    !tar -xf cryosparc_master.tar.gz cryosparc_master
    !rm /content/software/cryosparc/cryosparc_master.tar.gz
  if not os.path.isdir("cryosparc_worker"):
    if not os.path.exists("cryosparc_worker.tar.gz"):
      !curl -L https://get.cryosparc.com/download/worker-v4.7.1/$LICENSE_ID -o cryosparc_worker.tar.gz
    !tar -xf cryosparc_worker.tar.gz cryosparc_worker
    !rm /content/software/cryosparc/cryosparc_worker.tar.gz

/content/software
/content/software/cryosparc
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100  630M  100  630M    0     0  14.6M      0  0:00:43  0:00:43 --:--:-- 15.7M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:03 --:--:--     0
100 3993M  100 3993M    0     0  11.0M      0  0:06:02  0:06:02 --:--:-- 11.5M


The refinement process should be implemented in cryoSPARC as follows:

1. Import Micrographs.

2. CTF Estimation.

3. Import Particles (only coordinates).

4. Etract from Microgaphs.

5. Ab-initio Reconstruction.

6. Homogeneous Refinement.


In [11]:
# @title { display-mode: "form" }
# @markdown ## Install CryoSPARC

INSTALL = True  # @param {type:"boolean"}
EMAIL = "psudo@email.com"  # @param {type:"string"}
PASSWORD = "password"  # @param {type:"string"}
USERNAME = "Username"  # @param {type:"string"}
FIRST_NAME = "A"  # @param {type:"string"}
LAST_NAME = "B"  # @param {type:"string"}
PORT = 61000  # @param {type:"integer"}

import os

if INSTALL:
  %cd cryosparc_master
  !apt-get install iputils-ping -y
  %mkdir -p /content/software/cryosparc/cryosparc_cache

  os.environ['LICENSE_ID'] = LICENSE_ID
  os.environ['USER'] = USERNAME
  %env CRYOSPARC_FORCE_USER=true

  !./install.sh --standalone \
      --license $LICENSE_ID \
      --worker_path /content/software/cryosparc/cryosparc_worker \
      --ssdpath /content/software/cryosparc/cryosparc_cache \
      --initial_email "{EMAIL}" \
      --initial_password "{PASSWORD}" \
      --initial_username "{USERNAME}" \
      --initial_firstname "{FIRST_NAME}" \
      --initial_lastname "{LAST_NAME}" \
      --port {PORT} \
      --allowroot \
      --yes
  %cd {WORK_DIR}

/content/software/cryosparc/cryosparc_master
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  iputils-ping
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 43.0 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 iputils-ping amd64 3:20211215-1ubuntu0.1 [43.0 kB]
Fetched 43.0 kB in 1s (50.5 kB/s)
Selecting previously unselected package iputils-ping.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../iputils-ping_3%3a20211215-1ubuntu0.1_amd64.deb ...
Unpacking iputils-ping (3:20211215-1ubuntu0.1) ...
Setting up iputils-ping (3:20211215-1ubuntu0.1) ...
Processing triggers for man-db (2.10.2-1) ...
env: CRYOSPARC_FORCE_USER=true

************ CRYOSPARC SYSTEM: STANDALONE INSTALLER **************

You are the root user. Are you s

10017_01_50_02_100_watershed

In [12]:
# @title Set Cryosparc Parameter
pixel_size            = 1.77    # @param {type: "number"}
accel_voltage         = 300     # @param {type: "number"}
spherical_aberration  = 2.7     # @param {type: "number"}
total_exposure_dose   = 24      # @param {type: "number"}
amplitude_contrast    = 0.1     # @param {type: "number"}
# @markdown ---
symmetry              = "D2"    # @param {type: "string"}

In [13]:
# @title install `sparc-tools` package
# !pip install cryosparc-tools
!pip install "cryosparc-tools<5.0"

from cryosparc.tools import CryoSPARC

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.8 MB/s eta 0:00:00


In [14]:
# @title  { display-mode: "form" }
# @markdown Install pyngrok.

if INSTALL:
  %pip install pyngrok -qq

#### 🟪 Pyem Installation

In [15]:
# @title  { display-mode: "form" }

INSTALL = True # @param {type:"boolean"}

if INSTALL:
  %cd {SOFTWARE_DIR}
  !pip install pyFFTW
  !pip install healpy
  !pip install pathos
  !git clone https://github.com/asarnow/pyem.git
  %cd pyem
  !pip install --no-dependencies -e .
  %cd {WORK_DIR}

/content/software
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 

---

# Run CryoSparc

The refinement process should be implemented in cryoSPARC as follows:

1. Import Micrographs.

2. CTF Estimation.

3. Import Particles (only coordinates).

4. Extract from Microgaphs.

5. Ab-initio Reconstruction.

6. Homogeneous Refinement.


In [16]:
# !rsync -av CS-{PROJ_NAME} {RESULT_DIR}

---

# Cryosparc without GUI (all)

In [17]:
PROJ_NAME             = "test-00" # @param {type: "string"}


In [18]:
# @title Choose Desired Starfile
star_file_usage = "nms" # @param  ["cv", "tp", "nms"]
# @markdown ---
other_prefix_bool = True # @param {type: "boolean"}
if other_prefix_bool == True:
    other_star_prefix = "watershed" # @param {type: "string"}
    STAR_CHOSEN_DIR = STAR_DIR + f"/{other_star_prefix}_particles.star"
    !cp {STAR_DIR}/{other_star_prefix}_particles.star {EMPIAR_DIR}/
else:
    STAR_CHOSEN_DIR = STAR_DIR + f"/{star_file_usage}_particles.star"

cp: cannot stat '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_no_DT/10017/unet_eb5_dice_CRF/watershed_particles.star': No such file or directory


## Initial Activation Setup

In [19]:
# @title 0-0. Activate the cryosparc
start = True    # @param{type: "boolean"}
restart = False # @param{type: "boolean"}

# sparcm = f'/content/cryosparc/cryosparc_master/bin/cryosparcm'
sparcm = f'{SOFTWARE_DIR}/cryosparc/cryosparc_master/bin/cryosparcm'
if start == True:
    !{sparcm} start
if restart == True:
    !{sparcm} restart

Starting CryoSPARC System master process...
CryoSPARC is already running.
If you would like to restart, use cryosparcm restart


In [20]:
# @title View sparc status
status_view = True # @param{type: "boolean"}

if status_view == True:
    !{sparcm} status

----------------------------------------------------------------------------
CryoSPARC System master node installed at
/content/software/cryosparc/cryosparc_master
Current cryoSPARC version: v4.7.1
----------------------------------------------------------------------------

CryoSPARC process status:

app                              RUNNING   pid 5552, uptime 0:02:00
app_api                          RUNNING   pid 5581, uptime 0:01:58
app_api_dev                      STOPPED   Not started
command_core                     RUNNING   pid 5439, uptime 0:02:13
command_rtp                      RUNNING   pid 5499, uptime 0:02:05
command_vis                      RUNNING   pid 5493, uptime 0:02:06
database                         RUNNING   pid 5317, uptime 0:02:16

----------------------------------------------------------------------------
License is valid
----------------------------------------------------------------------------

global config variables:
export CRYOSPARC_LICENSE_ID="9657b74

---

In [21]:
# @title Print `list_projects()` in sparc

result = subprocess.run(
    [sparcm, 'cli', 'list_projects()'],
    capture_output=True,
    text=True,
    check=True
    )

sparc_projects_list_str = result.stdout
# String to List
import ast
sparc_projects_list = ast.literal_eval(sparc_projects_list_str)
sparc_projects_list

[]

In [22]:
# @title 0-1. Sparc Log in
hostname = socket.gethostname()

cs = CryoSPARC(
    license = LICENSE_ID,
    host = hostname,
    base_port = PORT,
    email = EMAIL,
    password = PASSWORD
)
assert cs.test_connection()

# ps = cs.api.projects.find()
# for p in ps:
#    print(f"UID: {p.uid} | Title: {p.title}")

Connection succeeded to CryoSPARC command_core at http://5c1395669406:61002
Connection succeeded to CryoSPARC command_vis at http://5c1395669406:61003
Connection succeeded to CryoSPARC command_rtp at http://5c1395669406:61005


In [23]:
# @title show all job types
show_all_job_types = True # @param{type: "boolean"}
if show_all_job_types == True:
    cs.print_job_types()

Section           | Job                              | Title                            
import            | import_movies                    | Import Movies                    
                  | import_micrographs               | Import Micrographs               
                  | import_particles                 | Import Particle Stack            
                  | import_volumes                   | Import 3D Volumes                
                  | import_templates                 | Import Templates                 
                  | import_result_group              | Import Result Group              
                  | import_beam_shift                | Import Beam Shift                
motion_correction | patch_motion_correction_multi    | Patch Motion Correction          
                  | rigid_motion_correction_multi    | Full-frame Motion Correction     
                  | local_motion_correction          | Local Motion Correction          
                  | l

In [24]:
# @title 0-2. Create a new sparc project
user_id = cs.cli.get_id_by_email(EMAIL)
# user_id = cs.api.users.me().id

 # ver 4.7 -
new_project = cs.cli.create_empty_project(
    owner_user_id = user_id,
    title = PROJ_NAME,
    project_container_dir = PROJECT_DIR
)


""" # ver 5.0 +
new_project = cs.api.projects.create(
    # owner_user_id = user_id,
    title = PROJ_NAME,
    parent_dir = PROJECT_DIR
)
"""

' # ver 5.0 +\nnew_project = cs.api.projects.create(\n    # owner_user_id = user_id,\n    title = PROJ_NAME,\n    parent_dir = PROJECT_DIR\n)\n'

---
## J1

In [25]:
# @title 1-1. Build job `import_micrographs`

# 1. Find your project
project = cs.find_project(new_project)

# 2. Build up workspace
workspace = project.create_workspace(title="workspace")

# 3. Build up job (here: import_micrographs)
micro_job = workspace.create_job("import_micrographs")

print(f"Job Built {micro_job.uid}")

Job Built J1


In [26]:
# @title 1-2. Set up Job J1 input parameters and J1 queue

default_dir_setting = True # @param{type: "boolean"}
# @markdown ---

# @markdown If you do not prefer to use the default setting of mrc directory:
if default_dir_setting == False:
    micrograph_dir = "" # @param{type: "string"}
    file_extension = "mrc" # @param{type: "string"}
    micrograph_dir = micrograph_dir + f"/*.{file_extension}"
else:
    micrograph_dir = f"/content/dataset/EMPIAR-{EMPIAR_ID}/micrographs/*.mrc"

# parameters setup
micro_job.set_param('blob_paths', micrograph_dir)
micro_job.set_param('psize_A', pixel_size)
micro_job.set_param('accel_kv', accel_voltage)
micro_job.set_param('cs_mm', spherical_aberration)
micro_job.set_param('total_dose_e_per_A2', total_exposure_dose)
micro_job.queue()

print(f"The Job {micro_job.uid} sent into the queue！")

# @markdown ---
micro_job_output_spec_show = False # @param{type: "boolean"}
if micro_job_output_spec_show == True:
    micro_job.print_output_spec()

# wait for finisted
print(f"Waiting for job {micro_job.uid} to finish...")
micro_job.wait_for_done(error_on_incomplete=True)
print(f"Job {micro_job.uid} has completed.")

The Job J1 sent into the queue！
Waiting for job J1 to finish...
Job J1 has completed.


In [27]:
# @title ⚠️ Clear the job(import micro), if any error happened

micro_job_clear = False # @param{type: "boolean"}

if micro_job_clear == True:
    micro_job.clear()

In [28]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(micro_job.uid)
    print(f"Current Job Status: {parent_job.status}")

Current Job Status: completed


---
## J2

In [29]:
# @title 2-1. Buid Job `Patch CTF Estimation`

ctf_job = workspace.create_job("ctf_estimation")
print(f"Job Built {ctf_job.uid}")

# Job Connection `imported_micrographs` -> `Patch CTF Estimation`
ctf_job.connect(
    target_input="exposures",
    source_job_uid=micro_job.uid,
    source_output="imported_micrographs",
)

Job Built J2


True

In [30]:
# @title `Patch CTF Estimation` job spec show
ctf_input_spec_show = False # @param{type: "boolean"}
if ctf_input_spec_show == True:
    ctf_job.print_input_spec()

ctf_param_spec_show = False # @param{type: "boolean"}
if ctf_param_spec_show == True:
    ctf_job.print_param_spec()

In [31]:
# @title 2-2. `Patch CTF Estimation` job execute

# CTF Estimation parameter setup
ctf_job.set_param('amplitude_contrast', amplitude_contrast)

ctf_job.queue(lane='default')
print(f"JOB CTF est. {ctf_job.uid} had sent into the queue！")


# wait for finisted
print(f"Waiting for job {ctf_job.uid} to finish...")
ctf_job.wait_for_done(error_on_incomplete=True)
print(f"Job {ctf_job.uid} has completed.")

JOB CTF est. J2 had sent into the queue！
Waiting for job J2 to finish...
Job J2 has completed.


In [32]:
# @title ⚠️ Clear the job(ctf), if any error happened

ctf_job_clear = False # @param{type: "boolean"}

if ctf_job_clear == True:
    ctf_job.clear()

In [33]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(ctf_job.uid)
    print(f"Current Job Status: {parent_job.status}")

Current Job Status: completed


---
## J3

### Import particles (only coordinates).

In [34]:
# @title 3-1. `import_particles` job section create
particles_job = workspace.create_job("import_particles")
print(f"Job Built {particles_job.uid}")

# connect with job ctf(J2)
particles_job.connect(
    target_input="micrographs",
    source_job_uid=ctf_job.uid,
    source_output="exposures_success",
)

Job Built J3


True

In [35]:
# @title `import_particles` job spec show
particles_input_spec_show = False # @param{type: "boolean"}
if particles_input_spec_show == True:
    particles_job.print_input_spec()

particles_param_spec_show = False # @param{type: "boolean"}
if particles_param_spec_show == True:
    particles_job.print_param_spec()

In [36]:
# @title 3-2. `import_particles` job execute

particles_job.set_param('particle_meta_path', STAR_CHOSEN_DIR)
particles_job.set_param('ignore_blob', True)
particles_job.set_param('remove_leading_uid', True)
particles_job.set_param('ignore_splits', True)

particles_job.set_param('accel_kv', accel_voltage)
particles_job.set_param('amp_contrast', amplitude_contrast)
particles_job.set_param('cs_mm', spherical_aberration)
particles_job.set_param('psize_A', pixel_size)

particles_job.set_param('enable_validation', True)  #strict check
particles_job.set_param('location_exists', True)    #Particle pick locations

particles_job.queue()
print(f" The Job import particles {particles_job.uid} had sent into the queue！")

# wait for finisted
print(f"Waiting for job {particles_job.uid} to finish...")
particles_job.wait_for_done(error_on_incomplete=True)
print(f"Job {particles_job.uid} has completed.")

CommandError: *** (http://5c1395669406:61002, code 400) Encountered ServerError from JSONRPC function "enqueue_job" with params {'project_uid': 'P1', 'job_uid': 'J3', 'lane': None, 'user_id': '69899fc4f4a438323eb5503a', 'hostname': None, 'gpus': False}:
ServerError: Failed to enqueue P1 J3 (import_particles): Job has builder errors. Particle meta path: Invalid path specified: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_no_DT/10017/unet_eb5_dice_CRF/watershed_particles.star directory allowed: False; file allowed: True; glob allowed: False
Traceback (most recent call last):
  File "/content/software/cryosparc/cryosparc_master/cryosparc_command/commandcommon.py", line 196, in wrapper
    res = func(*args, **kwargs)
  File "/content/software/cryosparc/cryosparc_master/cryosparc_command/commandcommon.py", line 265, in wrapper
    return func(*args, **kwargs)
  File "/content/software/cryosparc/cryosparc_master/cryosparc_co

In [ ]:
# @title ⚠️ Clear the job(particles), if any error happened

particles_job_clear = False # @param{type: "boolean"}

if particles_job_clear == True:
    particles_job.clear()

In [ ]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(particles_job.uid)
    print(f"Current Job Status: {parent_job.status}")

---

## J4


### Extract Micrographs

In [ ]:
# @title 4-1. `extract_micrographs` job section create
EMM_job = workspace.create_job("extract_micrographs_multi")
print(f"Job Built {EMM_job.uid}")

# connect with job ctf and particles
EMM_job.connect(
    target_input="micrographs",
    source_job_uid=ctf_job.uid,
    source_output="exposures_success",
)

EMM_job.connect(
    target_input="particles",
    source_job_uid=particles_job.uid,
    source_output="imported_particles",
)

In [ ]:
# @title `extract_micrographs` job spec show
EMM_input_spec_show = False # @param{type: "boolean"}
if EMM_input_spec_show == True:
    EMM_job.print_input_spec()

EMM_param_spec_show = False # @param{type: "boolean"}
if EMM_param_spec_show == True:
    EMM_job.print_param_spec()

In [ ]:
# @title 4-2. `extract_micrographs` job execute

EMM_job.queue("default")
print(f" The Job import particles {EMM_job.uid} had sent into the queue！")

# wait for finisted
print(f"Waiting for job {EMM_job.uid} to finish...")
EMM_job.wait_for_done(error_on_incomplete=True)
print(f"Job {EMM_job.uid} has completed.")

In [ ]:
# @title ⚠️ Clear the job(extract micro), if any error happened

EMM_job_clear = False # @param{type: "boolean"}

if EMM_job_clear == True:
    EMM_job.clear()

In [ ]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(EMM_job.uid)
    print(f"Current Job Status: {parent_job.status}")

---
## J5

### Ab-Initio Reconstruction

In [ ]:
# @title 5-1. `ab-initio` job section create
abinit_job = workspace.create_job("homo_abinit")
print(f"Job Built {abinit_job.uid}")

# connect with extracct micrograph

abinit_job.connect(
    target_input="particles",
    source_job_uid=EMM_job.uid,
    source_output="particles",
)


In [ ]:
# @title `ab-initio` job spec show
abinit_input_spec_show = False # @param{type: "boolean"}
if abinit_input_spec_show == True:
    abinit_job.print_input_spec()

abinit_param_spec_show = False # @param{type: "boolean"}
if abinit_param_spec_show == True:
    abinit_job.print_param_spec()

In [ ]:
# @title 5-2. `ab-initio` job execute

# set param
abinit_job.set_param('abinit_symmetry', symmetry)

abinit_job.queue("default")
print(f" The Job ab-initio {abinit_job.uid} had sent into the queue！")

# wait for finisted
print(f"Waiting for job {abinit_job.uid} to finish...")
abinit_job.wait_for_done(error_on_incomplete=True)
print(f"Job {abinit_job.uid} has completed.")

In [ ]:
# @title ⚠️ Clear the job(ab-initio), if any error happened

abinit_job_clear = False # @param{type: "boolean"}

if abinit_job_clear == True:
    abinit_job.clear()

In [ ]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(abinit_job.uid)
    print(f"Current Job Status: {parent_job.status}")

---
## J6

### Homogeneous Refinement

> 引用を追加



In [ ]:
# @title 6-1. `homogenous_refinement` job section create
homo_refin_job = workspace.create_job("homo_refine_new")
print(f"Job Built {homo_refin_job.uid}")

# connect with ab-initio
homo_refin_job.connect(
    target_input="particles",
    source_job_uid=abinit_job.uid,
    source_output="particles_all_classes",
)

homo_refin_job.connect(
    target_input="volume",
    source_job_uid=abinit_job.uid,
    source_output="volume_class_0",
)

opt_gp_CTF            = False    # @param {type: "boolean"}
opt_prtc_defocus      = False    # @param {type: "boolean"}

In [ ]:
# @title `homogenous_refinement` job spec show
homo_refin_input_spec_show = False # @param{type: "boolean"}
if homo_refin_input_spec_show == True:
    homo_refin_job.print_input_spec()

homo_refin_param_spec_show = False # @param{type: "boolean"}
if homo_refin_param_spec_show == True:
    homo_refin_job.print_param_spec()

In [ ]:
# @title 6-2. `homogenous_refinement` job execute

# set param
homo_refin_job.set_param('refine_symmetry', symmetry)
homo_refin_job.set_param('refine_ctf_global_refine', opt_gp_CTF)
homo_refin_job.set_param('refine_defocus_refine', opt_prtc_defocus)
homo_refin_job.queue("default")
print(f" The Job homogeneous refinement {homo_refin_job.uid} had sent into the queue！")

# wait for finisted
print(f"Waiting for job {homo_refin_job.uid} to finish...")
homo_refin_job.wait_for_done(error_on_incomplete=True)
print(f"Job {homo_refin_job.uid} has completed.")

In [ ]:
# @title ⚠️ Clear the job(homogeneous refinement), if any error happened

homo_refin_job_clear = False # @param{type: "boolean"}

if homo_refin_job_clear == True:
    homo_refin_job.clear()

In [ ]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(homo_refin_job.uid)
    print(f"Current Job Status: {parent_job.status}")

---
## j7


In [ ]:
# @title 7-1. `2d classification` job section create
o2d_class = workspace.create_job("class_2D_new")
print(f"Job Built {o2d_class.uid}")

o2d_class.connect(
    target_input="particles",
    source_job_uid=EMM_job.uid,
    source_output="particles",
)

In [ ]:
# @title `2d classification` job spec show
o2d_input_spec_show = False # @param{type: "boolean"}
if o2d_input_spec_show == True:
    o2d_class.print_input_spec()

o2d_param_spec_show = False # @param{type: "boolean"}
if o2d_param_spec_show == True:
    o2d_class.print_param_spec()

In [ ]:
# @title 7-2. `2d classification` job execute

# set param
o2d_class.set_param('class2D_K', 20)
o2d_class.queue("default")
print(f" The 2d classification {o2d_class.uid} had sent into the queue！")

# wait for finisted
print(f"Waiting for job {o2d_class.uid} to finish...")
o2d_class.wait_for_done(error_on_incomplete=True)
print(f"Job {o2d_class.uid} has completed.")

In [ ]:
# @title ⚠️ Clear the job( 2d classification ), if any error happened

o2d_class_job_clear = False # @param{type: "boolean"}

if o2d_class_job_clear == True:
    o2d_class.clear()

In [ ]:
# @title status print
status_print = True # @param{type: "boolean"}

if status_print == True:
    parent_job = project.find_job(o2d_class.uid)
    print(f"Current Job Status: {parent_job.status}")

---
## FSC Plot

In [ ]:
# @title functions for reading job.json and output FSC plot
def radwn_to_ang(radwn, N=None, psize=None, extent=None):
    """Converts wavenumber (radwn) to resolution in Angstroms."""
    if radwn == 0:
        return float('inf')
    if extent is None:
        return N * psize / radwn
    else:
        return extent / radwn

def gen_resolution_str_func(N, psize, radwnsq=False):
    """Generates a formatter function for matplotlib axes to display resolution in Angstroms."""
    def resolution_str(cradwn, pos):
        if radwnsq:
            cradwn = np.sqrt(cradwn)
        if cradwn > 0:
            res = radwn_to_ang(cradwn, N, psize)
            if res >= 100:
                # MODIFIED: Used raw string r'...' to prevent SyntaxWarning
                return r'{0:d}$\AA$'.format(int(res))
            elif res >= 10:
                 # MODIFIED: Used raw string r'...' to prevent SyntaxWarning
                return r'{0:.1f}$\AA$'.format(res)
            else:
                # MODIFIED: Used raw string r'...' to prevent SyntaxWarning
                return r'{0:.1f}$\AA$'.format(res)
        else:
            return 'DC'
    return resolution_str

def plot_fscs(radwn_max, fsc_info, N, psize, is_gold_standard=True):
    """Plots the FSC curves."""
    fig = plt.figure(figsize=(8, 6))
    ax = plt.subplot(111)

    # Dictionary to map fsc data keys to plot labels and resolution keys
    plots = {
        'fsc_nomask': ('No Mask', 'radwn_nomask_A'),
        'fsc_sphericalmask': ('Spherical', 'radwn_sphericalmask_A'),
        'fsc_loosemask': ('Loose', 'radwn_loosemask_A'),
        'fsc_tightmask': ('Tight', 'radwn_tightmask_A'),
        'fsc_noisesub': ('Corrected', 'radwn_noisesub_A')
    }

    # Plot each available FSC curve
    for fsc_key, (label_prefix, res_key) in plots.items():
        if hasattr(fsc_info, fsc_key) and getattr(fsc_info, fsc_key) is not None:
            res_val = getattr(fsc_info, res_key)
            if res_val:
                # MODIFIED: Used raw f-string fr'...' to prevent SyntaxWarning
                label = fr'{label_prefix} ({res_val:.1f}$\AA$)'
                plt.plot(fsc_info.radwns, getattr(fsc_info, fsc_key), label=label)

    plt.grid(True)
    plt.xlim(0, radwn_max)
    plt.ylim(0, 1)
    plt.axhline(0.143, color='k', linestyle='-', linewidth=1, label='FSC = 0.143')
    plt.legend(loc='best', fontsize=10)
    ax.xaxis.set_major_formatter(FuncFormatter(gen_resolution_str_func(N, psize)))

    res_final = fsc_info.radwn_final_A
    # MODIFIED: Used raw string r'...' to prevent SyntaxWarning
    title_str = ('GS' if is_gold_standard else '') + r'FSC Resolution: %.2f$\AA$' % res_final
    plt.title(title_str, fontsize=14)
    plt.xlabel("Spatial Frequency")
    plt.ylabel("Fourier Shell Correlation")

    return fig

# --- Main script ---

# A simple class to hold the parsed FSC data
class FSCData:
    def __init__(self, data, fallback_data):
        self.radwns = data.get('radwns')
        self.fsc_nomask = data.get('fsc_nomask')
        self.fsc_sphericalmask = fallback_data.get('fsc_sphericalmask')
        self.fsc_loosemask = data.get('fsc_loosemask')
        self.fsc_tightmask = data.get('fsc_tightmask')
        self.fsc_noisesub = data.get('fsc_noisesub')

        self.radwn_nomask_A = data.get('radwn_nomask_A')
        self.radwn_sphericalmask_A = fallback_data.get('radwn_sphericalmask_A')
        self.radwn_loosemask_A = data.get('radwn_loosemask_A')
        self.radwn_tightmask_A = data.get('radwn_tightmask_A')
        self.radwn_noisesub_A = data.get('radwn_noisesub_A')
        self.radwn_final_A = data.get('radwn_final_A')

In [ ]:
if __name__ == '__main__':
    # @title print out FSC plot
    save_dir_name = "FSC_plot_all" # @param{type:"string"}
    json_dir = f" {RESULT_DIR}/{save_dir_name}"
    !mkdir -p {RESULT_DIR}/{save_dir_name}
    homo_refin_job.download_file('job.json', f'{RESULT_DIR}/{save_dir_name}')

    file_name = "job"
    file_name = file_name + ".json"

    file_path = os.path.join(json_dir.strip(), file_name.strip())
    with open(file_path, 'r') as f:
        job_data = json.load(f)

    volume_group = next(g for g in job_data['output_result_groups'] if g['type'] == 'volume')
    fsc_stats = volume_group['latest_summary_stats']
    fsc_info_json = fsc_stats.get('fsc_info', {})
    fsc_info_best_json = fsc_stats.get('fsc_info_best', fsc_info_json)

    fsc_data_for_plot = FSCData(fsc_info_best_json, fsc_info_json)
    N = fsc_info_best_json.get('N', 256)
    psize = fsc_info_best_json.get('psize', 1.0)
    radwn_max = N / 2.0

    # Generate the plot
    if fsc_data_for_plot.radwns is not None:
        print("FSC data found. Generating plot...")
        fig = plot_fscs(radwn_max, fsc_data_for_plot, N, psize)

        # Save the plot to a file and display it
        fig.savefig(f'{json_dir.strip()}/fsc_plot.png', bbox_inches='tight', dpi=150)
        print("Plot saved as fsc_plot.png")
        plt.show()
    else:
        print("Error: Could not find 'radwns' array in the FSC data.")

---

---
## particles pose file(cs file) get (rotation angle)

In [ ]:
import os
import glob
import shutil
import re

# Parameters
tp_dir_name = f"CS-{PROJ_NAME.replace('_', '-')}"
base_path = "/content/sparc_project"
j6_path = os.path.join(base_path, tp_dir_name, "J6")

def get_latest_particle_file(source_dir, target_dir="/content"):
    # Target files like J6_004_particles.cs (Changed)
    search_pattern = os.path.join(source_dir, "J6_*_particles.cs")
    particle_files = glob.glob(search_pattern)

    if not particle_files:
        print(f"No particle files found in {source_dir}")
        return None

    def extract_iteration_number(filepath):
        filename = os.path.basename(filepath)
        match = re.search(r'_(\d+)', filename)
        if match:
            return int(match.group(1))
        return -1

    latest_file = max(particle_files, key=extract_iteration_number)

    dest_path = os.path.join(target_dir, os.path.basename(latest_file))
    shutil.copy2(latest_file, dest_path)

    print(f"Latest file found: {os.path.basename(latest_file)}")
    print(f"Successfully moved to {target_dir}")
    return dest_path

latest_path = get_latest_particle_file(j6_path)

In [ ]:
import os
import glob
import shutil
import re

# Parameters
tp_dir_name = f"CS-{PROJ_NAME.replace('_', '-')}"
base_path = "/content/sparc_project"
j6_path = os.path.join(base_path, tp_dir_name, "J6")

def get_sparc_files(source_dir, target_dir="/content"):
    # 1. Search for all iteration particle files
    search_pattern = os.path.join(source_dir, "J6_*_particles.cs")
    particle_files = [f for f in glob.glob(search_pattern) if "passthrough" not in f]

    if not particle_files:
        print(f"No particle files found in {source_dir}")
        return None, None

    # 2. Identify the latest iteration (e.g., J6_004)
    def extract_iteration_number(filepath):
        filename = os.path.basename(filepath)
        match = re.search(r'_(\d+)', filename)
        return int(match.group(1)) if match else -1

    latest_file = max(particle_files, key=extract_iteration_number)

    # 3. Identify the passthrough file (Changed)
    passthrough_file = os.path.join(source_dir, "J6_passthrough_particles.cs")

    # 4. Copy both to target_dir
    paths = []
    for f in [latest_file, passthrough_file]:
        if os.path.exists(f):
            dest = os.path.join(target_dir, os.path.basename(f))
            shutil.copy2(f, dest)
            paths.append(dest)
            print(f"Successfully moved {os.path.basename(f)} to {target_dir}")
        else:
            print(f"Warning: File not found: {f}")
            paths.append(None)

    return paths[0], paths[1] # latest_path, passthrough_path

# Execute retrieval
latest_path, pass_path = get_sparc_files(j6_path)

# --- Second Copy to RESULT_DIR ---
particles_rst_dir = os.path.join(RESULT_DIR, "particles_cs_rst")
os.makedirs(particles_rst_dir, exist_ok=True)

for path in [latest_path, pass_path]:
    if path:
        shutil.copy2(path, os.path.join(particles_rst_dir, os.path.basename(path)))
        print(f"Archived {os.path.basename(path)} in results directory.")

In [ ]:
!pip install mrcfile
!pip install starfile

In [ ]:
import mrcfile
import glob

# 1. Get the list of all micrographs
micrograph_pattern = os.path.join(f"{EMPIAR_DIR}/micrographs/", "*.mrc")
all_mrcs = glob.glob(micrograph_pattern)

if not all_mrcs:
    raise FileNotFoundError(f"No .mrc files found in {f"{EMPIAR_DIR}/micrographs/"}")

# 2. Pick the first one and read dimensions
with mrcfile.open(all_mrcs[0], header_only=True) as mrc:
    IMG_WIDTH = int(mrc.header.nx)
    IMG_HEIGHT = int(mrc.header.ny)

print(f"Detected Image Size: {IMG_WIDTH} x {IMG_HEIGHT}")

In [ ]:
import numpy as np
import pandas as pd

# Load the .cs data
res_data = np.load(latest_path)
pass_data = np.load(pass_path)

# 1. Align Passthrough to Refined Iteration Order
pass_uid_map = {uid: i for i, uid in enumerate(pass_data['uid'])}
# Map only UIDs present in the final refined set
aligned_indices = [pass_uid_map[uid] for uid in res_data['uid'] if uid in pass_uid_map]
aligned_pass = pass_data[aligned_indices]

# 2. Extract Poses and Fractional Coordinates
refined_poses = res_data['alignments3D/pose']
frac_x = aligned_pass['location/center_x_frac']
frac_y = aligned_pass['location/center_y_frac']

# 3. Convert Fractional to Pixel Coordinates (Changed)
# Pixel = Fraction * Dimension
pixel_x = frac_x * IMG_WIDTH
pixel_y = frac_y * IMG_HEIGHT

# 4. Consolidate into a final Result DataFrame
final_df = pd.DataFrame({
    'uid': res_data['uid'],
    'rlnCoordinateX': pixel_x,
    'rlnCoordinateY': pixel_y,
    'rlnImageName': [p.decode('utf-8').split('/')[-1] for p in aligned_pass['blob/path']],
    'pose_phi': refined_poses[:, 0],
    'pose_theta': refined_poses[:, 1],
    'pose_psi': refined_poses[:, 2]
})

def clean_image_name(name):
    # Regex finds the first underscore and takes everything after it
    match = re.search(r'_(.*)', name)
    return match.group(1) if match else name

final_df['rlnImageName'] = final_df['rlnImageName'].apply(clean_image_name)
final_df

# 5. Save to the Result Directory
csv_save_path = os.path.join(RESULT_DIR, "particles_cs_rst", "refined_particles_pixel_coords.csv")
# final_df.to_csv(csv_save_path, index=False)

print(f"Successfully processed {len(final_df)} particles.")
# print(f"Results saved to: {csv_save_path}")

In [ ]:
final_df

In [ ]:
import starfile
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import os

# 1. Final String Cleaning (Changed)
# Your current name: Falcon_2012..._particles.mrc
# Target name:      Falcon_2012....mrc
final_df['rlnImageName'] = final_df['rlnImageName'].str.replace('_particles', '', regex=False)

# 2. Load the original Star file
star_data = starfile.read(STAR_CHOSEN_DIR)
df_star = star_data[''] if isinstance(star_data, dict) else star_data

# 3. Efficient Spatial Matching (Changed)
tolerance = 2.0
matched_uids = []

# Grouping by micrograph for accuracy
star_groups = df_star.groupby('rlnMicrographName')
final_groups = final_df.groupby('rlnImageName')

print(f"Matching Cleaned Names. Sample: {final_df['rlnImageName'].iloc[0]}")

for mrc_name, final_group in final_groups:
    if mrc_name in star_groups.groups:
        star_group = star_groups.get_group(mrc_name)

        # Build KDTree for current micrograph coordinates
        star_coords = star_group[['rlnCoordinateX', 'rlnCoordinateY']].values
        tree = cKDTree(star_coords)

        # Query tree with refined coordinates
        final_coords = final_group[['rlnCoordinateX', 'rlnCoordinateY']].values
        d, i = tree.query(final_coords, distance_upper_bound=tolerance)

        # Filter for particles within 2-pixel range
        matches_in_group = final_group.iloc[d <= tolerance]
        matched_uids.extend(matches_in_group['uid'].tolist())

# 4. Filter and Save
final_matched_df = final_df[final_df['uid'].isin(matched_uids)].copy()
match_rate = (len(final_matched_df) / len(final_df)) * 100

print(f"Final Verification: {len(final_matched_df)}/{len(final_df)} matches ({match_rate:.2f}%)")

In [ ]:
import starfile
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import os

# 1. Load the original Star file
star_data = starfile.read(STAR_CHOSEN_DIR)
df_star = star_data[''] if isinstance(star_data, dict) else star_data

# 2. Match Refined Particles to Star File
tolerance = 3.0
matched_final_indices = []
matched_star_indices = []

# Group by micrograph for efficient spatial matching
star_groups = df_star.groupby('rlnMicrographName')
final_groups = final_df.groupby('rlnImageName')

for mrc_name, final_group in final_groups:
    if mrc_name in star_groups.groups:
        star_group = star_groups.get_group(mrc_name)

        # Build spatial index for Star picks in this micrograph
        star_coords = star_group[['rlnCoordinateX', 'rlnCoordinateY']].values
        tree = cKDTree(star_coords)

        # Query refined coordinates
        final_coords = final_group[['rlnCoordinateX', 'rlnCoordinateY']].values
        d, i = tree.query(final_coords, distance_upper_bound=tolerance)

        # Identify valid matches within 2 pixels
        valid_mask = d <= tolerance
        if valid_mask.any():
            # Get matching indices from both sides
            matched_final_indices.extend(final_group.index[valid_mask].tolist())
            # tree index 'i' refers to the positional index in star_group
            matched_star_indices.extend(star_group.index[i[valid_mask]].tolist())

# Create matched subsets
final_matched = final_df.loc[matched_final_indices].copy()
star_matched = df_star.loc[matched_star_indices].copy()

# Temporarily link them to ensure they sort identically
final_matched['_match_idx'] = range(len(final_matched))
star_matched['_match_idx'] = range(len(star_matched))

In [ ]:
# 3. Ascending Sort by Name, X, and Y (changed)
# Primary sort on final_df
final_sorted = final_matched.sort_values(
    by=['rlnImageName', 'rlnCoordinateX', 'rlnCoordinateY'],
    ascending=True
)

# Reorder star_matched to mirror final_sorted perfectly
star_sorted = star_matched.iloc[final_sorted['_match_idx'].values].copy()

# Clean up temporary alignment columns
final_sorted.drop(columns=['_match_idx'], inplace=True)
star_sorted.drop(columns=['_match_idx'], inplace=True)

# 4. Storage to Result Directory
os.makedirs(os.path.join(RESULT_DIR, "particles_dfcsv_rst"), exist_ok=True)
final_sorted.to_csv(os.path.join(RESULT_DIR, "particles_dfcsv_rst", "angle_df_ascending.csv"), index=False)
star_sorted.to_csv(os.path.join(RESULT_DIR, "particles_dfcsv_rst", "star_data_ascending.csv"), index=True)

print(f"Successfully aligned and sorted {len(final_sorted)} particles in ascending order.")

In [ ]:
final_sorted

In [ ]:
star_sorted

In [ ]:
# import time

# time.sleep(10000000)

---
## 2d class

In [ ]:
import os
import glob
import shutil
import re
import numpy as np
import pandas as pd

# 1. Path Configuration (Auto-detecting J7)
tp_dir_name = f"CS-{PROJ_NAME.replace('_', '-')}"
base_path = "/content/sparc_project"
j7_source_dir = os.path.join(base_path, tp_dir_name, "J7")

def get_latest_j7_files(source_dir):
    # Find all iteration files, excluding passthrough
    search_pattern = os.path.join(source_dir, "J7_*_particles.cs")
    particle_files = [f for f in glob.glob(search_pattern) if "passthrough" not in f]

    if not particle_files:
        print(f"No particle files found in {source_dir}")
        return None, None

    # Identify latest iteration number
    def extract_iteration(filepath):
        match = re.search(r'_(\d+)', os.path.basename(filepath))
        return int(match.group(1)) if match else -1

    latest_res_path = max(particle_files, key=extract_iteration)
    passthrough_path = os.path.join(source_dir, "J7_passthrough_particles.cs")

    return latest_res_path, passthrough_path

# 2. Automated Execution
j7_latest_path, j7_pass_path = get_latest_j7_files(j7_source_dir)

if j7_latest_path and os.path.exists(j7_pass_path):
    # Load the data
    res_data_2d = np.load(j7_latest_path)
    pass_data_2d = np.load(j7_pass_path)

    # Align Passthrough to J7 Iteration Order
    pass_uid_map = {uid: i for i, uid in enumerate(pass_data_2d['uid'])}
    aligned_indices = [pass_uid_map[uid] for uid in res_data_2d['uid'] if uid in pass_uid_map]
    aligned_pass_2d = pass_data_2d[aligned_indices]

    # 3. Consolidate into DataFrame (changed)
    class_df = pd.DataFrame({
        'uid': res_data_2d['uid'],
        'class_id': res_data_2d['alignments2D/class'],
        'rlnImageName': [p.decode('utf-8').split('/')[-1] for p in aligned_pass_2d['location/micrograph_path']],
        'rlnCoordinateX': aligned_pass_2d['location/center_x_frac'] * IMG_WIDTH,
        'rlnCoordinateY': aligned_pass_2d['location/center_y_frac'] * IMG_HEIGHT
    })

    # Clean Image Names
    class_df['rlnImageName'] = class_df['rlnImageName'].apply(lambda x: re.search(r'_(.*)', x).group(1) if "_" in x else x)

    class_df_sorted = class_df.sort_values(
    by=['rlnImageName', 'rlnCoordinateX', 'rlnCoordinateY'],
    ascending=True
    )

    # 4. Automated Storage in RESULT_DIR (changed)
    output_dir = os.path.join(RESULT_DIR, "particles_dfcsv_rst")
    os.makedirs(output_dir, exist_ok=True)

    csv_save_path = os.path.join(output_dir, "class.csv")
    class_df_sorted.to_csv(csv_save_path, index=False)

    print(f"Successfully processed {len(class_df_sorted)} particles.")
    print(f"File stored at: {csv_save_path}")
else:
    print("Failed to locate required J7 files.")

class_df.head()

In [ ]:
import numpy as np

# This returns an array where each entry is the class ID for that particle
class_ids = res_data_2d['alignments2D/class']

# Print the first 10 assignments
print(class_ids[:10])

# To see how many particles are in each class:
unique, counts = np.unique(class_ids, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count} particles")

---

# Run CryoSparc

In [ ]:
ngrok = False # @param{type:"boolean"}

In [ ]:
if ngrok:
    from pyngrok import ngrok, conf
    import getpass

In [ ]:
if ngrok:
    print("Enter your authtoken, which can be copied from https://dashboard.ngrok.com/")
    conf.get_default().auth_token = getpass.getpass()

In [ ]:
if ngrok:
    # Setup a tunnel to the port 61000
    public_url = ngrok.connect(61000)

In [ ]:
if ngrok:
    print(public_url)